<a href="https://colab.research.google.com/github/warrensuca/chinese-recipe-api/blob/main/warrensuca/chinese-recipe-api/scraper/the_woks_of_life_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#The Woks of Life Recipe Scraper
This notebook scrapes recipe data from the Woks of Life website: https://thewoksoflife.com/category/recipes/:

For each recipe, we collect:

Name
- Ingredients
- Procedure
- Nutrition Facts

The final dataset is stored as a Pandas DataFrame and exported to CSV.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display
import time
print("setup complete")

setup complete


In [ ]:
session = requests.Session()

session.headers.update({
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
})

print("Session initialized")

Session initialized


#Scrape individual recipe URL
Exploration. Later, we will turn this into a function

In [ ]:

url = "https://thewoksoflife.com/ayam-kecap-indonesian-braised-chicken/"
headings = [
      "Name",
      "Ingredients",
      "Procedure",
      "Nutrition_Facts"
  ]

response = session.get(url, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

recipe_data = {
    "Name": url.split("/")[-2] #getting title is an additional call, just clean later
}
print(recipe_data["Name"])
print('\n')

ingredients_list_raw = soup.find_all("li", class_ = "wprm-recipe-ingredient")
ingredients_list = []

for ingredient in ingredients_list_raw:
  ingredients_list.append(ingredient.get_text()[2:]) #remove a bullet point svg

print(ingredients_list)
print('\n')
recipe_data["Ingredients"] = ingredients_list

procedure_list_raw = soup.find_all("li", class_ = "wprm-recipe-instruction")
procedure_list = []

for procedure in procedure_list_raw:
  procedure_list.append(procedure.get_text())

print(procedure_list)
print('\n')
recipe_data["Procedure"] = procedure_list

nutrition_facts_list_raw = soup.find_all("span", class_ = "wprm-nutrition-label-text-nutrition-container")
nutrition_facts = []
for nutrition_fact in nutrition_facts_list_raw:
  #print(nutrition_fact.get_text())
  nutrition_facts.append(nutrition_fact.get_text())

print(nutrition_facts)

recipe_data["Nutrition_Facts"] = nutrition_facts

ayam-kecap-indonesian-braised-chicken


['2½ pounds chicken drumsticks', '½ teaspoon salt', '2 tablespoons neutral oil (such as canola, vegetable, or avocado oil)', '½ teaspoon ginger (finely minced or grated)', '3 cloves garlic (chopped)', '1/3 cup shallot or red onion (finely diced)', '3 dried chili peppers (optional)', '2¼ cups low sodium chicken stock', '¾ teaspoon galangal (or sand ginger powder)', '¾ teaspoon coriander powder', '¾ teaspoon ground or whole cloves', '3 dried Makrut lime leaves', '3 tablespoons kecap manis (Indonesian sweet soy sauce)', '2 tablespoons soy sauce', '1 teaspoon tamarind paste (optional)', '½ teaspoon ground white or black pepper']


['In a large bowl, add the chicken legs, and sprinkle them with the salt. Rub the salt in evenly, and let sit for at least 10 minutes. Heat a medium pot over medium-high heat, and add the oil. Sear the chicken legs until lightly browned on all sides. Remove the chicken to a plate.', 'Add the ginger, garlic, shallot/onion, a

#Create function based off exploration
This function will be used on each recipe of the site

In [ ]:
def scrape_recipe(url: str):


  response = session.get(url, timeout=10)
  response.raise_for_status()

  soup = BeautifulSoup(response.text, "html.parser")

  recipe_data = {
      "Name": url.split("/")[-2] #getting title is an additional call, just clean later
  }


  ingredients_list_raw = soup.find_all("li", class_ = "wprm-recipe-ingredient")
  ingredients_list = []

  for ingredient in ingredients_list_raw:
    ingredients_list.append(ingredient.get_text()[2:]) #remove a bullet point svg


  recipe_data["Ingredients"] = ingredients_list

  procedure_list_raw = soup.find_all("li", class_ = "wprm-recipe-instruction")
  procedure_list = []

  for procedure in procedure_list_raw:
    procedure_list.append(procedure.get_text())



  recipe_data["Procedure"] = procedure_list

  nutrition_facts_list_raw = soup.find_all("span", class_ = "wprm-nutrition-label-text-nutrition-container")
  nutrition_facts = []
  for nutrition_fact in nutrition_facts_list_raw:
    #print(nutrition_fact.get_text())
    nutrition_facts.append(nutrition_fact.get_text())

  recipe_data["Nutrition_Facts"] = nutrition_facts

  return recipe_data

In [ ]:
recipe_data = scrape_recipe("https://thewoksoflife.com/pork-belly-char-siu/")

recipe_data

{'Name': 'pork-belly-char-siu',
 'Ingredients': ['2 pounds boneless skinless pork belly (if trimming your pork belly at home, save the bones and skins for soup!)',
  '3 tablespoons granulated sugar',
  '1½ teaspoons salt',
  '¼ teaspoon white pepper',
  '¼ teaspoon five spice powder',
  '1 tablespoon hoisin sauce',
  '1 tablespoon Shaoxing wine',
  '2 teaspoons light soy sauce',
  '½ teaspoon sesame oil',
  '1 clove garlic (minced)',
  '4 to 5 drops of red food coloring (optional)',
  '1 tablespoon maltose or honey',
  '1 tablespoon hot water'],
 'Procedure': ['If you have a larger piece of pork belly, cut it lengthwise into long 1½- to 2-inch (4-5cm) thick strips.',
  'Make the marinade in a large bowl by combining the sugar, salt, white pepper, five spice powder, hoisin sauce, Shaoxing wine, light soy sauce, sesame oil, and minced garlic. Add the pork belly, and use your hands to work the marinade into the strips. Cover and marinate in the refrigerator overnight.',
  'Arrange a rack 

#Scrape single page
Exploration

In [ ]:

response = session.get("https://thewoksoflife.com/category/recipes/page/{}/".format(1), timeout=10)
response.raise_for_status()

data = []

soup = BeautifulSoup(response.text, "html.parser")

recipes = soup.find_all(class_ = "wp-block-post-featured-image")


for recipe in recipes[:5]:

  data.append(scrape_recipe(recipe.a.get("href")))
data

[{'Name': 'pork-belly-char-siu',
  'Ingredients': ['2 pounds boneless skinless pork belly (if trimming your pork belly at home, save the bones and skins for soup!)',
   '3 tablespoons granulated sugar',
   '1½ teaspoons salt',
   '¼ teaspoon white pepper',
   '¼ teaspoon five spice powder',
   '1 tablespoon hoisin sauce',
   '1 tablespoon Shaoxing wine',
   '2 teaspoons light soy sauce',
   '½ teaspoon sesame oil',
   '1 clove garlic (minced)',
   '4 to 5 drops of red food coloring (optional)',
   '1 tablespoon maltose or honey',
   '1 tablespoon hot water'],
  'Procedure': ['If you have a larger piece of pork belly, cut it lengthwise into long 1½- to 2-inch (4-5cm) thick strips.',
   'Make the marinade in a large bowl by combining the sugar, salt, white pepper, five spice powder, hoisin sauce, Shaoxing wine, light soy sauce, sesame oil, and minced garlic. Add the pork belly, and use your hands to work the marinade into the strips. Cover and marinate in the refrigerator overnight.',
  

In [ ]:
def scrape_page(page_num: int):

  response = session.get("https://thewoksoflife.com/category/recipes/page/{}/".format(page_num), timeout=10)
  response.raise_for_status()

  data = []

  soup = BeautifulSoup(response.text, "html.parser")

  recipes = soup.find_all(class_ = "wp-block-post-featured-image")


  for recipe in recipes:

    data.append(scrape_recipe(recipe.a.get("href")))

  return data

In [ ]:
page_one_data = scrape_page(1)

page_one_data

[{'Name': 'pork-belly-char-siu',
  'Ingredients': ['2 pounds boneless skinless pork belly (if trimming your pork belly at home, save the bones and skins for soup!)',
   '3 tablespoons granulated sugar',
   '1½ teaspoons salt',
   '¼ teaspoon white pepper',
   '¼ teaspoon five spice powder',
   '1 tablespoon hoisin sauce',
   '1 tablespoon Shaoxing wine',
   '2 teaspoons light soy sauce',
   '½ teaspoon sesame oil',
   '1 clove garlic (minced)',
   '4 to 5 drops of red food coloring (optional)',
   '1 tablespoon maltose or honey',
   '1 tablespoon hot water'],
  'Procedure': ['If you have a larger piece of pork belly, cut it lengthwise into long 1½- to 2-inch (4-5cm) thick strips.',
   'Make the marinade in a large bowl by combining the sugar, salt, white pepper, five spice powder, hoisin sauce, Shaoxing wine, light soy sauce, sesame oil, and minced garlic. Add the pork belly, and use your hands to work the marinade into the strips. Cover and marinate in the refrigerator overnight.',
  

In [ ]:
columns = [
      "Name",
      "Ingredients",
      "Procedure",
      "Nutrition_Facts"
  ]

out = []

for page_num in range(1, 3):
  data = scrape_page(page_num)
  out.extend(data)
print(out)
df = pd.DataFrame(out, columns=columns)

print(f"Total recipes scraped: {len(df)}")

display(df.head())

[{'Name': 'pork-belly-char-siu', 'Ingredients': ['2 pounds boneless skinless pork belly (if trimming your pork belly at home, save the bones and skins for soup!)', '3 tablespoons granulated sugar', '1½ teaspoons salt', '¼ teaspoon white pepper', '¼ teaspoon five spice powder', '1 tablespoon hoisin sauce', '1 tablespoon Shaoxing wine', '2 teaspoons light soy sauce', '½ teaspoon sesame oil', '1 clove garlic (minced)', '4 to 5 drops of red food coloring (optional)', '1 tablespoon maltose or honey', '1 tablespoon hot water'], 'Procedure': ['If you have a larger piece of pork belly, cut it lengthwise into long 1½- to 2-inch (4-5cm) thick strips.', 'Make the marinade in a large bowl by combining the sugar, salt, white pepper, five spice powder, hoisin sauce, Shaoxing wine, light soy sauce, sesame oil, and minced garlic. Add the pork belly, and use your hands to work the marinade into the strips. Cover and marinate in the refrigerator overnight.', 'Arrange a rack in the middle of the oven and

,Name,Ingredients,Procedure,Nutrition_Facts
0,pork-belly-char-siu,[2 pounds boneless skinless pork belly (if tri...,"[If you have a larger piece of pork belly, cut...","[Calories: 682kcal (34%), Carbohydrates: 11g (..."
1,red-bean-zongzi,[2 pounds sweet rice (also known as sticky ric...,"[In a large, fine mesh strainer, rinse the swe...","[Calories: 344kcal (17%), Carbohydrates: 72g (..."
2,stir-fry-spinach,[1 pound spinach (preferably Taiwanese spinach...,[Wash spinach clean by soaking a few times and...,"[Calories: 130kcal (7%), Carbohydrates: 5g (2%..."
3,gluten-free-scallion-pancakes,[15 rice paper wrappers (that you'd use for Vi...,[Fill a 9-inch pie plate (or other shallow dis...,"[Calories: 428kcal (21%), Carbohydrates: 57g (..."
4,st-paul-sandwich,"[4 to 6 cups neutral oil (such as vegetable, c...","[If you’re making the gravy, make it first. He...","[Calories: 480kcal (24%), Carbohydrates: 35g (..."


#Scrape the entire site!

In [ ]:
pages = 80

out = []

for page_num in range(1, pages + 1):



    try:
      data = scrape_page(page_num)
      out.extend(data)

    except Exception as e:

      print(f"Error on page {page_num}: {e}")
      break
    print(f"page number: {page_num}")
    time.sleep(4) #be nice to the site, too aggresive can cause an exception


page number: 1
page number: 2
page number: 3
page number: 4
page number: 5
page number: 6
page number: 7
page number: 8
page number: 9
page number: 10
page number: 11
page number: 12
page number: 13
page number: 14
page number: 15
page number: 16
page number: 17
page number: 18
page number: 19
page number: 20
page number: 21
page number: 22
page number: 23
page number: 24
page number: 25
page number: 26
page number: 27
page number: 28
page number: 29
page number: 30
page number: 31
page number: 32
page number: 33
page number: 34
page number: 35
page number: 36
page number: 37
page number: 38
page number: 39
page number: 40
page number: 41
page number: 42
page number: 43
page number: 44
page number: 45
page number: 46
page number: 47
page number: 48
page number: 49
page number: 50
page number: 51
page number: 52
page number: 53
page number: 54
page number: 55
page number: 56
page number: 57
page number: 58
page number: 59
page number: 60
page number: 61
page number: 62
page number: 63
p